In [1]:
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import tensorflow
from tensorflow.compat.v1.keras.backend import get_session
tensorflow.compat.v1.disable_v2_behavior()
import kerasAC 
from kerasAC.generators.tiledb_predict_generator import *
from kerasAC.tiledb_config import *
import tiledb 
from scipy.special import softmax
from kerasAC.interpret.deepshap import * 
from kerasAC.interpret.profile_shap import * 
from kerasAC.vis import * 
from kerasAC.helpers.transform_bpnet_io import * 

Instructions for updating:
non-resource variables are not supported in the long term


Using TensorFlow backend.
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:173: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  min_coords = np.vstack(data.min(0) for data in polygons_data).min(0)
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:174: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  max_coords = np.vstack(data.max(0) for data in polygons_data).max(0)


In [2]:
gen=TiledbPredictGenerator(ref_fasta="/mnt/data/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                    batch_size=200,
                    bed_regions_center='summit',
                    bed_regions="HEPG2.dnase.idr.optimal_peak.narrowPeak.gz",
                    tdb_partition_thresh_for_upsample=None,
                    tdb_partition_attribute_for_upsample=None,
                    tdb_array="/srv/scratch/annashch/encode_dnase_tiledb/db/dnase",
                    chrom_sizes="/users/annashch/hg38.chrom.sizes",
                    tdb_input_flank=[673],
                    tdb_input_source_attribute=["seq"],
                    tdb_input_aggregation=[None],
                    tdb_input_transformation=[None],
                    tdb_output_source_attribute=["count_bigwig_unstranded_5p","count_bigwig_unstranded_5p"],
                    tdb_output_flank=[500,500],
                    tdb_output_aggregation=[None,"sum"],
                    tdb_output_transformation=[None,"log"],
                    num_inputs=1,
                    num_outputs=2,
                    upsample_ratio=None,
                    tasks=['ENCSR149XIL'],
                    tdb_ambig_attribute=None,
                    tdb_config=get_default_config(),
                    tdb_ctx=tiledb.Ctx(config=get_default_config()),
                    num_threads=1)
                    

opening:/srv/scratch/annashch/encode_dnase_tiledb/db/dnase for reading...
success!
got indices for used chroms
identified task indices:[3]
loaded bed regions
created generator
created predict generator


In [3]:
#load the model! 
from keras.models import load_model
from keras.utils.generic_utils import get_custom_objects
from kerasAC.metrics import * 
from kerasAC.custom_losses import * 
custom_objects={"recall":recall,
                    "sensitivity":recall,
                    "specificity":specificity,
                    "fpr":fpr,
                    "fnr":fnr,
                    "precision":precision,
                    "f1":f1,
                    "ambig_binary_crossentropy":ambig_binary_crossentropy,
                    "ambig_mean_absolute_error":ambig_mean_absolute_error,
                    "ambig_mean_squared_error":ambig_mean_squared_error,
                    "MultichannelMultinomialNLL":MultichannelMultinomialNLL}
get_custom_objects().update(custom_objects)


In [ ]:
label_prof_dict={} 
label_count_dict={} 
pred_prof_dict={} 
pred_count_dict={} 
profile_shap_dict={}
count_shap_dict={}
seq_dict={}

for fold in range(1):
    model=load_model("/srv/scratch/annashch/deeplearning/profile/hepg2_dnase/seed.1234.cw.101.filters.500.naive.range.4.6.to.11.5/seed.1234.cs.101.filters.500.naive.range.4.6.to.11.5.bias.observed."+str(fold)+".hdf5")
    model_wrapper=(model.input, model.layers[-1].output)
    count_explainer=shap.DeepExplainer(model_wrapper,data=create_background,combine_mult_and_diffref=combine_mult_and_diffref_1d)
    prof_explainer = create_explainer(model)
    length_gen=len(gen)
    for i in range(length_gen): 
        print(str(i)+'/'+str(length_gen))
        X,y,coords=gen[i]
        coords=[[entry.decode('utf8')  for entry in coord] for coord in coords]
        preds=model.predict(X)

        pred_prob=get_probability_track_from_bpnet(preds[0])
        label_prob=get_probability_label_track(y[0].squeeze())
        
        label_sum=y[1].squeeze() 
        pred_sum=preds[1].squeeze()         
        
        profile_explanations=prof_explainer(X[0],None,None)
        count_explanations=np.squeeze(count_explainer.shap_values(X)[0])
        #store outputs in dictionary 
        for coord_index in range(len(coords)): 
            cur_coord=coords[coord_index][0:2]
            cur_coord[1]=int(cur_coord[1])
            cur_coord=tuple(cur_coord)
            label_prof_dict[cur_coord]=label_prob[coord_index]
            label_count_dict[cur_coord]=label_sum[coord_index]
            pred_prof_dict[cur_coord]=pred_prob[coord_index]
            pred_count_dict[cur_coord]=pred_sum[coord_index]
            profile_shap_dict[cur_coord]=profile_explanations[coord_index,:]
            count_shap_dict[cur_coord]=count_explanations[coord_index,:]
            seq_dict[cur_coord]=X[0][coord_index]
        


Instructions for updating:
If using Keras pass *_constraint arguments to layers.
HERE--tf
HERE--keras
here!
here!
0/679


1/679
2/679
3/679
4/679
5/679
6/679
7/679
8/679
9/679
10/679
11/679
12/679
13/679
14/679
15/679
16/679
17/679
18/679
19/679
20/679
21/679
22/679
23/679
24/679
25/679
26/679
27/679
28/679
29/679
30/679
31/679
32/679
33/679
34/679
35/679
36/679
37/679
38/679
39/679
40/679
41/679
42/679
43/679
44/679
45/679
46/679
47/679
48/679
49/679
50/679
51/679
52/679
53/679
54/679
55/679
56/679
57/679
58/679
59/679
60/679
61/679
62/679
63/679
64/679
65/679
66/679
67/679
68/679
69/679
70/679
71/679
72/679
73/679
74/679
75/679
76/679
77/679
78/679
79/679
80/679
81/679
82/679
83/679
84/679
85/679
86/679
87/679
88/679
89/679
90/679
91/679
92/679
93/679
94/679
95/679
96/679
97/679
98/679
99/679
100/679
101/679
102/679
103/679
104/679
105/679
106/679
107/679
108/679
109/679
110/679
111/679
112/679
113/679
114/679
115/679
116/679
117/679
118/679
119/679
120/679
121/679
122/679
123/679
124/679
125/679
126/679
127/679
128/679
129/679
130/679
131/679
132/679
133/679
134/679
135/679
136/679
137/679
138/679
139/

In [12]:
i

27

In [ ]:
#save the dictionaries to disk! 
import pickle
outputs={} 
outputs['label_prof']=label_prof_dict
outputs['label_sum']=label_count_dict
outputs['pred_prof']=pred_prof_dict
outputs['pred_sum']=pred_count_dict
outputs['profile_shap']=profile_shap_dict 
outputs['count_shap']=count_shap_dict 
outputs['seq']=seq_dict 
pickle.dump(outputs,open("HEPG2.DNASE.baseline.fold0.deepSHAP", "wb" ) )

In [13]:
outputs['count_shap'][('chr2',42504936)].shape

(1346, 4)

In [18]:
len(list(set(outputs['count_shap'].keys())))

135628

In [ ]:
## for plotting 
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import matplotlib 
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"]=10,5
plt.rcParams['axes.xmargin'] = 0

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}

matplotlib.rc('font', **font)

In [ ]:
plot_seq_importance(outputs['count_shap'][('chr2',42504936)][173+400:173+600,:],outputs['seq'][('chr2',42504936)][173+400:173+600,:],figsize=(15,2))


In [ ]:
plot_seq_importance(outputs['profile_shap'][('chr2',42504936)][173+400:173+600,:],outputs['seq'][('chr2',42504936)][173+400:173+600,:],figsize=(15,2))


In [ ]:
plt.plot(outputs['label_prof'][('chr2',42504936)][400:600])
plt.plot(outputs['pred_prof'][('chr2',42504936)][400:600])
